# NHRT — Complete Research Implementation with TorchTune
### Nested Hierarchical Reasoning Transformer · Qwen2.5-0.5B · GSM8K

**Full pipeline:** Architecture inspection → Module design → Integration → TorchTune fine-tuning → LoRA control → Multi-layer → SVAMP generalisation → Statistical analysis → Checkpoint export

This notebook is the authoritative, end-to-end implementation. It begins from the very first experiment (model loading and architecture inspection from Notebook v1), continues through every validated finding (HRB, gated NestedDecoderLayer, layer-12 winner from Phase 1), and extends the research with TorchTune's supervised fine-tuning recipe replacing the raw AdamW loop — giving gradient clipping, LR scheduling, and proper logging for free.

**Phase 1 results already established (do not re-run ablation from scratch):**
```
Baseline (n=60): 3.33%   Layer 6: 10.00%   Layer 12: 11.67% ← winner
Layer 18: 8.33%          Layer 23: 5.00%
```

## 1. Install Dependencies

In [1]:
!pip install -q transformers accelerate sentencepiece datasets peft scipy
!pip install -q torchtune
# torchtune needs torch >= 2.2. Colab T4 ships 2.3+ so this is fine.
print("All packages installed.")

All packages installed.


## 2. Imports & Global Configuration

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import re, gc, json, csv, time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# ── constants ──────────────────────────────────────────────────────────────────
MODEL_NAME   = "Qwen/Qwen2.5-0.5B"
HIDDEN_SIZE  = 896
TARGET_LAYER = 12          # winner from Phase 1 ablation
N_TRAIN      = 200         # scale up: was 60 in Phase 1
N_EVAL       = 100         # scale up: was 60
EPOCHS       = 3
LR           = 2e-4
MAX_SEQ_LEN  = 512
CHECKPOINT   = "nhrt_torchtune_final.pt"
RESULTS_CSV  = "nhrt_results.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")
try:
    import torchtune
    print(f"TorchTune: {torchtune.__version__}")
except Exception as e:
    print(f"TorchTune import note: {e}")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device : cuda
PyTorch: 2.10.0+cu128
TorchTune: 0.6.1+cpu


## 3. Architecture Inspection (from Notebook v1)
Loading the base model once and systematically documenting every component — the foundation of the entire research.

In [3]:
# Load once — never reload unnecessarily
# NOTE: no device_map="auto" — this is a 0.5B model, no need to shard it
# across multiple GPUs, and doing so breaks the layer-swap below (accelerate's
# per-layer dispatch hooks conflict with the manually-.to()'d NestedDecoderLayer,
# producing "Expected all tensors to be on the same device" errors).
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto"
).to(device)

total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model            : {MODEL_NAME}")
print(f"Total parameters : {total_params:,}")
print(f"Decoder layers   : {len(baseline_model.model.layers)}")
print(f"Hidden size      : {baseline_model.config.hidden_size}")
print(f"Vocab size       : {baseline_model.config.vocab_size}")
print(f"Num attn heads   : {baseline_model.config.num_attention_heads}")
print(f"Num KV heads     : {baseline_model.config.num_key_value_heads} (GQA)")
print(f"Intermediate dim : {baseline_model.config.intermediate_size}")
print(f"Dtype            : {baseline_model.dtype}")
print()
print("── Layer 0 (representative decoder layer) ──")
print(baseline_model.model.layers[0])

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model            : Qwen/Qwen2.5-0.5B
Total parameters : 494,032,768
Decoder layers   : 24
Hidden size      : 896
Vocab size       : 151936
Num attn heads   : 14
Num KV heads     : 2 (GQA)
Intermediate dim : 4864
Dtype            : torch.bfloat16

── Layer 0 (representative decoder layer) ──
Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [4]:
# Tensor trace: text → embedding → RMSNorm → Q/K/V shapes
probe_text   = "What is 25 + 17?"
probe_inputs = tokenizer(probe_text, return_tensors="pt").to(device)

layer0       = baseline_model.model.layers[0]
embeddings   = baseline_model.model.embed_tokens(probe_inputs["input_ids"])
hidden_norm  = layer0.input_layernorm(embeddings)

q = layer0.self_attn.q_proj(hidden_norm)
k = layer0.self_attn.k_proj(hidden_norm)
v = layer0.self_attn.v_proj(hidden_norm)

print(f"Input tokens : {probe_inputs['input_ids'].shape}")
print(f"Embeddings   : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"After RMSNorm: {hidden_norm.shape}")
print(f"Q            : {q.shape}  (14 heads × 64 dim)")
print(f"K            : {k.shape}  (2 KV heads × 64 dim — GQA 7:1)")
print(f"V            : {v.shape}")
print()
print("── MLP (SwiGLU) ──")
print(layer0.mlp)

Input tokens : torch.Size([1, 10])
Embeddings   : torch.Size([1, 10, 896])  dtype=torch.bfloat16
After RMSNorm: torch.Size([1, 10, 896])
Q            : torch.Size([1, 10, 896])  (14 heads × 64 dim)
K            : torch.Size([1, 10, 128])  (2 KV heads × 64 dim — GQA 7:1)
V            : torch.Size([1, 10, 128])

── MLP (SwiGLU) ──
Qwen2MLP(
  (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
  (up_proj): Linear(in_features=896, out_features=4864, bias=False)
  (down_proj): Linear(in_features=4864, out_features=896, bias=False)
  (act_fn): SiLUActivation()
)


## 4. Reasoning Module Definitions
Three modules discovered and validated across Notebooks v1–v3. Defined exactly once.

In [5]:
# ── Module 1: NestedReasoningModule (NRM) ─────────────────────────────────────
# Original design from Notebook v1. Three parallel linear branches + aggregation.
# Superseded by HRB but kept for ablation completeness.
class NestedReasoningModule(nn.Module):
    """Parallel multi-branch reasoner (v1 original design)."""
    def __init__(self, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.reasoner1  = nn.Linear(hidden_size, hidden_size)
        self.reasoner2  = nn.Linear(hidden_size, hidden_size)
        self.reasoner3  = nn.Linear(hidden_size, hidden_size)
        self.aggregate  = nn.Linear(hidden_size * 3, hidden_size)
        self.activation = nn.GELU()

    def forward(self, x):
        r1 = self.activation(self.reasoner1(x))
        r2 = self.activation(self.reasoner2(x))
        r3 = self.activation(self.reasoner3(x))
        return self.aggregate(torch.cat([r1, r2, r3], dim=-1))


# ── Module 2: HierarchicalReasoningBlock (HRB) ────────────────────────────────
# Best-performing module from Phase 1 ablation (layer 12: 11.67% vs 3.33% baseline).
# Gated blend of fast (shallow) and deep (multi-layer) reasoning paths.
class HierarchicalReasoningBlock(nn.Module):
    """Gated fast+deep hierarchical reasoning (Phase 1 winner)."""
    def __init__(self, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.fast_reasoner = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.GELU()
        )
        self.deep_reasoner = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2), nn.GELU(),
            nn.Linear(hidden_size * 2, hidden_size * 2), nn.GELU(),
            nn.Linear(hidden_size * 2, hidden_size)
        )
        self.controller = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size), nn.Sigmoid()
        )

    def forward(self, x):
        fast = self.fast_reasoner(x)
        deep = self.deep_reasoner(x)
        gate = self.controller(torch.cat([fast, deep], dim=-1))
        return gate * fast + (1 - gate) * deep


# ── Module 3: DeepReasoner ────────────────────────────────────────────────────
# Universal-Transformer-style iterative residual loop.
# RMSNorm added in v2 for training stability.
class DeepReasoner(nn.Module):
    """Iterative residual reasoning loop (nested-loop concept)."""
    def __init__(self, hidden_size=HIDDEN_SIZE, steps=3):
        super().__init__()
        self.steps        = steps
        self.reason_block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.GELU(),
            nn.Linear(hidden_size, hidden_size)
        )
        self.norm = nn.RMSNorm(hidden_size, eps=1e-6)

    def forward(self, x):
        state = x
        for _ in range(self.steps):
            state = state + self.norm(self.reason_block(state))
        return state


# ── Sanity check ──────────────────────────────────────────────────────────────
test_x = hidden_norm.detach()
for name, mod in [("NRM", NestedReasoningModule()), ("HRB", HierarchicalReasoningBlock()), ("DeepReasoner", DeepReasoner())]:
    m   = mod.to(device).to(test_x.dtype)
    out = m(test_x)
    n   = sum(p.numel() for p in m.parameters())
    shift = (out - test_x).abs().mean().item()
    print(f"{name:16s} | shape={tuple(out.shape)} | params={n:,} | shift={shift:.5f}")

NRM              | shape=(1, 10, 896) | params=4,820,480 | shift=0.06641
HRB              | shape=(1, 10, 896) | params=8,837,248 | shift=0.06592
DeepReasoner     | shape=(1, 10, 896) | params=1,608,320 | shift=1.75781


## 5. NestedDecoderLayer — The Core NHRT Contribution
Wraps a Qwen2DecoderLayer, inserting the reasoning module via a learnable gated residual:
`hidden = hidden + gate × RMSNorm(reasoning_module(hidden))`

The gate starts near-zero (0.05) so the model initialises as almost-identical to baseline. Training opens the gate only if the reasoning module provides useful signal.

In [6]:
class NestedDecoderLayer(nn.Module):
    """
    Inserts a reasoning module between Attention and MLP in a Qwen2 decoder layer.

    Forward pass:
        residual = hidden
        x        = input_layernorm(hidden)
        attn_out = self_attn(x, **kwargs)
        hidden   = residual + attn_out

        ← NHRT: hidden = hidden + gate * RMSNorm(reasoning_module(hidden))

        residual = hidden
        x        = post_attention_layernorm(hidden)
        hidden   = residual + mlp(x)
    """
    def __init__(self, original_layer, reasoning_module, init_gate: float = 0.05):
        super().__init__()
        # Unwrap if already wrapped — prevents double-nesting on re-patch
        self.original          = original_layer.original if hasattr(original_layer, "original") else original_layer
        self.reasoning_module  = reasoning_module
        hidden_size            = self.original.self_attn.q_proj.in_features  # 896
        self.reason_norm       = nn.RMSNorm(hidden_size, eps=1e-6)
        self.reason_gate       = nn.Parameter(torch.tensor(init_gate, dtype=torch.float32))

    def __getattr__(self, name):
        # Proxy attribute lookups the outer HF model expects on a decoder layer
        # (e.g. `attention_type`, used by newer `transformers` versions to build
        # the causal-mask mapping) through to the wrapped original layer, since
        # this wrapper doesn't otherwise inherit from Qwen2DecoderLayer.
        try:
            return super().__getattr__(name)
        except AttributeError:
            original = self.__dict__.get("_modules", {}).get("original")
            if original is not None and hasattr(original, name):
                return getattr(original, name)
            raise

    def forward(self, hidden_states, **kwargs):
        # ── 1. Attention sublayer ──────────────────────────────────────────────
        residual      = hidden_states
        x             = self.original.input_layernorm(hidden_states)
        attn_out, *_  = self.original.self_attn(hidden_states=x, **kwargs)
        hidden_states = residual + attn_out

        # ── 2. NHRT Reasoning injection ───────────────────────────────────────
        r_out         = self.reasoning_module(hidden_states)
        r_out         = self.reason_norm(r_out)
        hidden_states = hidden_states + self.reason_gate * r_out

        # ── 3. MLP sublayer ───────────────────────────────────────────────────
        residual      = hidden_states
        x             = self.original.post_attention_layernorm(hidden_states)
        hidden_states = residual + self.original.mlp(x)

        return hidden_states


# ── Integration test ──────────────────────────────────────────────────────────
seq_len           = hidden_norm.shape[1]
position_ids      = torch.arange(seq_len, device=device).unsqueeze(0)
attention_mask    = torch.ones((1, seq_len), device=device)
position_emb      = baseline_model.model.rotary_emb(hidden_norm, position_ids)

_hrb_test         = HierarchicalReasoningBlock().to(device).to(hidden_norm.dtype)
_test_layer       = NestedDecoderLayer(baseline_model.model.layers[0], _hrb_test).to(device).to(hidden_norm.dtype)
_test_out         = _test_layer(hidden_norm, position_ids=position_ids,
                                position_embeddings=position_emb, attention_mask=attention_mask)

print(f"NestedDecoderLayer output : {_test_out.shape}  dtype={_test_out.dtype}")
print(f"Gate init value           : {_test_layer.reason_gate.item():.4f}")
del _hrb_test, _test_layer, _test_out

NestedDecoderLayer output : torch.Size([1, 10, 896])  dtype=torch.bfloat16
Gate init value           : 0.0500


## 6. Evaluation Helpers

In [7]:
def extract_gsm8k_answer(text: str):
    """Prefer #### marker (GSM8K ground-truth format), fall back to last number."""
    m = re.search(r"####\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None


def generate_answer(model, question: str, max_new_tokens: int = 64) -> str:
    if not question or not question.strip():
        # An empty prompt tokenizes to 0 tokens and crashes model.generate()
        # (IndexError in _cache_dependant_input_preparation). Fail soft instead.
        return ""
    inputs = tokenizer(question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def run_eval(model, dataset, max_new_tokens=64, desc="Eval", return_preds=False):
    model.eval()
    correct, preds = 0, []
    for sample in tqdm(dataset, desc=desc, leave=False):
        pred_text = generate_answer(model, sample["question"], max_new_tokens)
        pred_num  = extract_gsm8k_answer(pred_text)
        true_num  = extract_gsm8k_answer(sample["answer"])
        ok        = (pred_num == true_num)
        correct  += int(ok)
        preds.append(ok)
    acc = correct / len(dataset)
    return (acc, preds) if return_preds else acc


def mcnemar_test(preds_a, preds_b, label_a="A", label_b="B"):
    """Exact McNemar's test using binomial (continuity-corrected for n<25)."""
    from scipy.stats import binomtest
    b = sum(1 for a, b_ in zip(preds_a, preds_b) if a and not b_)
    c = sum(1 for a, b_ in zip(preds_a, preds_b) if not a and b_)
    n = b + c
    if n == 0:
        print(f"{label_a} vs {label_b}: no disagreements — identical predictions.")
        return None
    result = binomtest(c, n, p=0.5, alternative="two-sided")
    winner = label_b if c > b else label_a
    sig    = "✓ significant (p<0.05)" if result.pvalue < 0.05 else "✗ not significant"
    print(f"{label_a} vs {label_b}: {label_a}-only={b}  {label_b}-only={c}  "
          f"p={result.pvalue:.4f}  {sig}  (favours {winner})")
    return result.pvalue

print("Helpers defined.")

Helpers defined.


## 7. Load Datasets

In [8]:
gsm8k = load_dataset("openai/gsm8k", name="main", trust_remote_code=True)
svamp = load_dataset("garrethlee/svamp", name="default", trust_remote_code=True)   # same answer format — free generalisation check

gsm_train = gsm8k["train"].select(range(N_TRAIN))
gsm_eval  = gsm8k["test"].select(range(N_EVAL))

# SVAMP: use first N_EVAL samples from test split
svamp_eval = svamp["test"].select(range(N_EVAL))

print(gsm8k)
print()
print(f"gsm_train : {len(gsm_train)} samples")
print(f"gsm_eval  : {len(gsm_eval)} samples")
print(f"svamp_eval: {len(svamp_eval)} samples")
print()
# Confirm SVAMP answer key name
print("SVAMP sample keys:", list(svamp_eval[0].keys()))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openai/gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/67.5k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

gsm_train : 200 samples
gsm_eval  : 100 samples
svamp_eval: 100 samples

SVAMP sample keys: ['question', 'answer']


## 8. Baseline Evaluation
Establishing the reference point on both benchmarks before any training.

In [9]:
baseline_gsm_acc, baseline_gsm_preds = run_eval(
    baseline_model, gsm_eval, desc="Baseline GSM8K", return_preds=True
)

# SVAMP (garrethlee/svamp) already ships {question, answer} columns —
# no Body/Question/Answer split like the original SVAMP release. Handle
# both schemas defensively and skip any sample that still ends up empty
# (an empty string tokenizes to 0 tokens and crashes model.generate()).
def svamp_sample_to_gsm(sample):
    """Normalise SVAMP sample to {question, answer} format."""
    if "question" in sample:
        q = sample["question"]
    else:
        q = (sample.get("Body", "") + " " + sample.get("Question", "")).strip()
    a = str(sample.get("answer", sample.get("Answer", "")))
    return {"question": q.strip(), "answer": a}

svamp_gsm_eval = [svamp_sample_to_gsm(s) for s in svamp_eval]
# Guard: drop any sample whose question ended up empty (would crash generate())
svamp_gsm_eval = [s for s in svamp_gsm_eval if s["question"]]
baseline_svamp_acc = run_eval(
    baseline_model, svamp_gsm_eval, desc="Baseline SVAMP"
)

print()
print("=" * 50)
print("BASELINE RESULTS")
print("=" * 50)
print(f"GSM8K  (n={N_EVAL}): {baseline_gsm_acc:.2%}")
print(f"SVAMP  (n={N_EVAL}): {baseline_svamp_acc:.2%}")

Baseline GSM8K:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 9. TorchTune-Style Training Loop
TorchTune's `FullModelTorchTuneWrapper` and recipe classes require a config YAML and CLI invocation when used formally. On Colab we instead use TorchTune's building blocks directly in Python:
- **`torchtune.modules`**: `RMSNorm`, cosine LR scheduler
- **`torchtune.training`**: gradient clipping, loss utilities
- **Custom `NHRTSFTDataset`**: formats GSM8K as `question\nanswer` causal-LM targets

This gives all the TorchTune recipe benefits (grad clipping, warmup-cosine LR, per-step logging) without needing a config file.

In [ ]:
# TorchTune components
try:
    from torchtune.training import get_cosine_schedule_with_warmup
    from torchtune.modules import RMSNorm as TTRMSNorm
    HAS_TORCHTUNE = True
    print("Using TorchTune scheduler and RMSNorm.")
except ImportError:
    HAS_TORCHTUNE = False
    print("TorchTune not available — falling back to torch equivalents.")

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Dataset wrapper ────────────────────────────────────────────────────────────
class NHRTSFTDataset(torch.utils.data.Dataset):
    """
    Formats GSM8K samples as 'question\nanswer' strings for causal LM fine-tuning.
    Labels are set to -100 for the question tokens (train only on the answer).
    """
    def __init__(self, dataset, tokenizer, max_length=MAX_SEQ_LEN, mask_question=True):
        self.data         = dataset
        self.tokenizer    = tokenizer
        self.max_length   = max_length
        self.mask_question = mask_question

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        question = sample["question"]
        answer   = sample["answer"]
        full_text = question + "\n" + answer

        q_enc  = self.tokenizer(question, add_special_tokens=True)
        q_len  = len(q_enc["input_ids"])

        enc = self.tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        input_ids = enc["input_ids"].squeeze(0)

        labels = input_ids.clone()
        if self.mask_question:
            labels[:q_len] = -100    # mask question — supervise only answer tokens

        return {"input_ids": input_ids, "labels": labels}


def collate_fn(batch):
    max_len = max(b["input_ids"].shape[0] for b in batch)
    input_ids_list, labels_list = [], []
    for b in batch:
        pad_len = max_len - b["input_ids"].shape[0]
        input_ids_list.append(F.pad(b["input_ids"], (0, pad_len), value=tokenizer.pad_token_id))
        labels_list.append(F.pad(b["labels"],    (0, pad_len), value=-100))
    return {
        "input_ids": torch.stack(input_ids_list),
        "labels":    torch.stack(labels_list)
    }


# ── Core training function (TorchTune-style) ──────────────────────────────────
def train_nhrt(model, train_dataset, epochs=EPOCHS, lr=LR,
               grad_clip=1.0, warmup_ratio=0.1, batch_size=1,
               desc="Training"):
    """
    TorchTune-inspired SFT loop with:
      - Answer-only loss masking (labels[:q_len] = -100)
      - Cosine LR with linear warmup
      - Gradient clipping (grad_clip=1.0)
      - Per-epoch loss logging
    """
    sft_ds     = NHRTSFTDataset(train_dataset, tokenizer)
    loader     = torch.utils.data.DataLoader(sft_ds, batch_size=batch_size,
                                             shuffle=True, collate_fn=collate_fn)
    optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01)

    total_steps  = epochs * len(loader)
    warmup_steps = int(total_steps * warmup_ratio)

    if HAS_TORCHTUNE:
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=lr * 0.1)

    model.train()
    step = 0
    history = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch in tqdm(loader, desc=f"{desc} E{epoch+1}/{epochs}", leave=False):
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)

            outputs   = model(input_ids=input_ids, labels=labels)
            loss      = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], grad_clip
            )
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step       += 1

        avg = epoch_loss / len(loader)
        history.append(avg)
        print(f"  [{desc}] Epoch {epoch+1}/{epochs}  avg_loss={avg:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    return history

print("TorchTune training utility ready.")

## 10. Build the NHRT Model (Layer 12 — Phase 1 Winner)

In [ ]:
nhrt_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto"
).to(device)

hrb          = HierarchicalReasoningBlock(HIDDEN_SIZE).to(device).to(nhrt_model.dtype)
nested_layer = NestedDecoderLayer(nhrt_model.model.layers[TARGET_LAYER], hrb,
                                  init_gate=0.05).to(device).to(nhrt_model.dtype)
nhrt_model.model.layers[TARGET_LAYER] = nested_layer

# Freeze base, train only HRB + gate
for p in nhrt_model.parameters():
    p.requires_grad = False
for p in nhrt_model.model.layers[TARGET_LAYER].reasoning_module.parameters():
    p.requires_grad = True
nhrt_model.model.layers[TARGET_LAYER].reason_gate.requires_grad = True

trainable = sum(p.numel() for p in nhrt_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in nhrt_model.parameters())

# Functional check
with torch.no_grad():
    _inp = tokenizer("What is 3 + 4?", return_tensors="pt").to(device)
    _out = nhrt_model.generate(**_inp, max_new_tokens=8, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    print("Functional check :", tokenizer.decode(_out[0], skip_special_tokens=True))

print(f"Layer {TARGET_LAYER} patched with HRB.")
print(f"Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")
print(f"Init gate: {nhrt_model.model.layers[TARGET_LAYER].reason_gate.item():.4f}")

## 11. Train NHRT with TorchTune SFT Loop

In [ ]:
nhrt_history = train_nhrt(
    nhrt_model, gsm_train,
    epochs=EPOCHS, lr=LR,
    desc=f"NHRT Layer{TARGET_LAYER}"
)

trained_gate = nhrt_model.model.layers[TARGET_LAYER].reason_gate.item()
print(f"\nTrained gate: {trained_gate:.4f}  (started at 0.0500)")
print(f"Gate delta  : {trained_gate - 0.05:+.4f}  "
      f"({'gate opened = HRB was useful' if trained_gate > 0.05 else 'gate closed = HRB not used'})")

## 12. Post-Training Evaluation: NHRT vs Baseline

In [ ]:
nhrt_gsm_acc, nhrt_gsm_preds = run_eval(
    nhrt_model, gsm_eval, desc="NHRT GSM8K", return_preds=True
)

# SVAMP generalisation — zero additional training, purely testing transfer
nhrt_svamp_acc = run_eval(
    nhrt_model, svamp_gsm_eval, desc="NHRT SVAMP"
)

print()
print("=" * 55)
print("NHRT vs BASELINE — POST-TRAINING RESULTS")
print("=" * 55)
print(f"{'Metric':<30}{'Baseline':>10}{'NHRT':>10}{'Δ pp':>10}")
print("-" * 55)
print(f"{'GSM8K accuracy':<30}{baseline_gsm_acc:>10.2%}{nhrt_gsm_acc:>10.2%}"
      f"{(nhrt_gsm_acc-baseline_gsm_acc)*100:>+9.2f}")
print(f"{'SVAMP accuracy':<30}{baseline_svamp_acc:>10.2%}{nhrt_svamp_acc:>10.2%}"
      f"{(nhrt_svamp_acc-baseline_svamp_acc)*100:>+9.2f}")
print()
print(f"Trained gate value: {trained_gate:.4f}")

In [ ]:
# Statistical significance
print("\nMcNemar's test (paired per-question correctness):")
mcnemar_test(nhrt_gsm_preds, baseline_gsm_preds, "NHRT", "Baseline")

In [ ]:
# Logit-level analysis
with torch.no_grad():
    _inp  = tokenizer("What is 25 + 17?", return_tensors="pt").to(device)
    b_log = baseline_model(**_inp).logits.float()
    n_log = nhrt_model(**_inp).logits.float()

mse     = F.mse_loss(b_log, n_log)
cos_sim = F.cosine_similarity(b_log.reshape(1,-1), n_log.reshape(1,-1))
abs_del = (b_log - n_log).abs().mean()

print(f"Logit MSE              : {mse.item():.6f}")
print(f"Logit cosine similarity: {cos_sim.item():.6f}")
print(f"Mean absolute delta    : {abs_del.item():.6f}")

In [ ]:
# Qualitative spot-check — 3 questions
for i in range(3):
    q = gsm_eval[i]["question"]
    gt = extract_gsm8k_answer(gsm_eval[i]["answer"])
    b_ans  = generate_answer(baseline_model, q)
    nh_ans = generate_answer(nhrt_model, q)
    print("="*70)
    print("Q :", q[:120])
    print("GT:", gt)
    print("Base:", extract_gsm8k_answer(b_ans), " | Full:", b_ans[-60:])
    print("NHRT:", extract_gsm8k_answer(nh_ans), " | Full:", nh_ans[-60:])

## 13. Inference Overhead Measurement

In [ ]:
_inp = tokenizer("What is 100 divided by 4?", return_tensors="pt").to(device)

REPS = 10
def timed_forward(m):
    times = []
    with torch.no_grad():
        for _ in range(REPS):
            t0 = time.perf_counter()
            _ = m(**_inp)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return sum(times[2:]) / len(times[2:])   # drop 2 warmup runs

base_t = timed_forward(baseline_model)
nhrt_t = timed_forward(nhrt_model)
overhead = (nhrt_t - base_t) / base_t * 100

print(f"Baseline forward pass : {base_t*1000:.2f} ms")
print(f"NHRT forward pass     : {nhrt_t*1000:.2f} ms")
print(f"Overhead              : {overhead:+.2f}%")

## 14. Parameter-Matched LoRA Control
**The most important experiment:** if a LoRA adapter with the same trainable-parameter budget
at the same layer beats NHRT, the gain is just "more parameters", not the HRB architecture.

In [ ]:
!pip install -U torchao
!pip install -U peft transformers accelerate

In [ ]:



# LoRA target: layer 12's projection matrices only
# r=32 across 7 matrices ≈ 8.8M trainable params (matches HRB budget)
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto"
).to(device)

lora_targets = [
    f"model.layers.{TARGET_LAYER}.self_attn.q_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.k_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.v_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.o_proj",
    f"model.layers.{TARGET_LAYER}.mlp.gate_proj",
    f"model.layers.{TARGET_LAYER}.mlp.up_proj",
    f"model.layers.{TARGET_LAYER}.mlp.down_proj",
]

lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=lora_targets,
    lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
lora_history = train_nhrt(
    lora_model, gsm_train,
    epochs=EPOCHS, lr=LR,
    desc=f"LoRA Layer{TARGET_LAYER}"
)

lora_gsm_acc, lora_gsm_preds = run_eval(
    lora_model, gsm_eval, desc="LoRA GSM8K", return_preds=True
)
lora_svamp_acc = run_eval(lora_model, svamp_gsm_eval, desc="LoRA SVAMP")

print(f"LoRA GSM8K : {lora_gsm_acc:.2%}")
print(f"LoRA SVAMP : {lora_svamp_acc:.2%}")

del lora_model
gc.collect(); torch.cuda.empty_cache()

## 15. Multi-Layer NHRT — Layers [6, 12, 18]
All three individual layers showed positive signal in Phase 1. This tests compounding vs interference.

In [ ]:
MULTI_LAYERS = [6, 12, 18]

multi_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto"
).to(device)

for l_idx in MULTI_LAYERS:
    hrb_l   = HierarchicalReasoningBlock(HIDDEN_SIZE).to(device).to(multi_model.dtype)
    nested  = NestedDecoderLayer(multi_model.model.layers[l_idx], hrb_l,
                                  init_gate=0.05).to(device).to(multi_model.dtype)
    multi_model.model.layers[l_idx] = nested

for p in multi_model.parameters():
    p.requires_grad = False
for l_idx in MULTI_LAYERS:
    for p in multi_model.model.layers[l_idx].reasoning_module.parameters():
        p.requires_grad = True
    multi_model.model.layers[l_idx].reason_gate.requires_grad = True

multi_trainable = sum(p.numel() for p in multi_model.parameters() if p.requires_grad)
print(f"Multi-layer trainable: {multi_trainable:,}  (~{multi_trainable/trainable:.1f}x single-layer)")

In [ ]:
multi_history = train_nhrt(
    multi_model, gsm_train,
    epochs=EPOCHS, lr=LR * 0.5,   # lower LR for multi-layer stability
    desc="NHRT Multi-Layer"
)

multi_gsm_acc, multi_gsm_preds = run_eval(
    multi_model, gsm_eval, desc="Multi-NHRT GSM8K", return_preds=True
)
multi_svamp_acc = run_eval(multi_model, svamp_gsm_eval, desc="Multi-NHRT SVAMP")

print()
for l_idx in MULTI_LAYERS:
    g = multi_model.model.layers[l_idx].reason_gate.item()
    print(f"Layer {l_idx} gate: {g:.4f}  (delta: {g-0.05:+.4f})")

del multi_model
gc.collect(); torch.cuda.empty_cache()

## 16. DeepReasoner Variant at Layer 12
Direct comparison of HRB vs DeepReasoner (the "nested loop" concept) with identical training.

In [ ]:
deep_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto"
).to(device)

dr         = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(deep_model.dtype)
dr_layer   = NestedDecoderLayer(deep_model.model.layers[TARGET_LAYER], dr,
                                 init_gate=0.05).to(device).to(deep_model.dtype)
deep_model.model.layers[TARGET_LAYER] = dr_layer

for p in deep_model.parameters():
    p.requires_grad = False
for p in deep_model.model.layers[TARGET_LAYER].reasoning_module.parameters():
    p.requires_grad = True
deep_model.model.layers[TARGET_LAYER].reason_gate.requires_grad = True

dr_params = sum(p.numel() for p in deep_model.parameters() if p.requires_grad)
print(f"DeepReasoner trainable: {dr_params:,}")

In [ ]:

deep_history = train_nhrt(
    deep_model, gsm_train,
    epochs=EPOCHS, lr=LR,
    desc="DeepReasoner"
)

deep_gsm_acc, deep_gsm_preds = run_eval(
    deep_model, gsm_eval, desc="DeepReasoner GSM8K", return_preds=True
)
deep_svamp_acc = run_eval(deep_model, svamp_gsm_eval, desc="DeepReasoner SVAMP")
deep_gate = deep_model.model.layers[TARGET_LAYER].reason_gate.item()

print(f"DeepReasoner GSM8K : {deep_gsm_acc:.2%}")
print(f"DeepReasoner SVAMP : {deep_svamp_acc:.2%}")
print(f"DeepReasoner gate  : {deep_gate:.4f}")

del deep_model
gc.collect(); torch.cuda.empty_cache()

## 17. Full Results Summary

In [ ]:
print("=" * 75)
print("NHRT COMPLETE RESULTS — ALL CONFIGURATIONS")
print("=" * 75)
print(f"{'Configuration':<32}{'GSM8K':>10}{'SVAMP':>10}{'Δ GSM8K':>12}{'Δ SVAMP':>12}")
print("-" * 75)

rows = [
    ("Baseline (frozen)",    baseline_gsm_acc, baseline_svamp_acc),
    (f"LoRA Layer{TARGET_LAYER} (~matched params)", lora_gsm_acc, lora_svamp_acc),
    (f"NHRT HRB Layer{TARGET_LAYER} (TorchTune)", nhrt_gsm_acc, nhrt_svamp_acc),
    ("NHRT HRB Multi [6,12,18]", multi_gsm_acc, multi_svamp_acc),
    ("NHRT DeepReasoner L12",   deep_gsm_acc, deep_svamp_acc),
]

for name, gsm, sv in rows:
    dg = (gsm - baseline_gsm_acc) * 100
    ds = (sv  - baseline_svamp_acc) * 100
    print(f"{name:<32}{gsm:>10.2%}{sv:>10.2%}{dg:>+11.2f}pp{ds:>+11.2f}pp")

print()
print("Architecture insight:")
print(f"  NHRT vs LoRA (same params): {'NHRT wins — architecture matters' if nhrt_gsm_acc > lora_gsm_acc else 'LoRA wins — need stronger HRB design'}")
print(f"  Multi vs Single layer     : {'Multi compounds — synergy across depth' if multi_gsm_acc > nhrt_gsm_acc else 'Single wins — layer interference detected'}")
print(f"  HRB vs DeepReasoner       : {'HRB wins' if nhrt_gsm_acc > deep_gsm_acc else 'DeepReasoner wins — iterative loop is stronger'}")

In [ ]:
print("\nStatistical Significance (McNemar's test):")
mcnemar_test(nhrt_gsm_preds,   baseline_gsm_preds, "NHRT",         "Baseline")
mcnemar_test(nhrt_gsm_preds,   lora_gsm_preds,     "NHRT",         "LoRA")
mcnemar_test(multi_gsm_preds,  nhrt_gsm_preds,     "Multi-NHRT",   "Single-NHRT")
mcnemar_test(deep_gsm_preds,   nhrt_gsm_preds,     "DeepReasoner", "HRB")

## 18. Save Checkpoint & Results CSV

In [ ]:
# Reload nhrt_model for saving (may have been gc'd if OOM — rebuild)
try:
    gate_check = nhrt_model.model.layers[TARGET_LAYER].reason_gate.item()
    torch.save(nhrt_model.state_dict(), CHECKPOINT)
    print(f"Saved checkpoint: {CHECKPOINT}  (gate={gate_check:.4f})")
except Exception as e:
    print(f"Checkpoint save skipped: {e}")

# CSV results
results_rows = [
    {"config": "Baseline",               "gsm8k": baseline_gsm_acc, "svamp": baseline_svamp_acc},
    {"config": f"LoRA_L{TARGET_LAYER}",  "gsm8k": lora_gsm_acc,     "svamp": lora_svamp_acc},
    {"config": f"NHRT_HRB_L{TARGET_LAYER}", "gsm8k": nhrt_gsm_acc,  "svamp": nhrt_svamp_acc},
    {"config": "NHRT_Multi_6_12_18",     "gsm8k": multi_gsm_acc,    "svamp": multi_svamp_acc},
    {"config": "NHRT_DeepReasoner_L12",  "gsm8k": deep_gsm_acc,     "svamp": deep_svamp_acc},
]
with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["config","gsm8k","svamp"])
    writer.writeheader()
    writer.writerows(results_rows)

print(f"Saved results  : {RESULTS_CSV}")
print()
for r in results_rows:
    print(f"  {r['config']:<35} GSM8K={r['gsm8k']:.2%}  SVAMP={r['svamp']:.2%}")

## 19. Gate Sensitivity Analysis (Post-Training)
After training, sweep the gate value to see the accuracy response curve.
This answers: "what gate value does the trained HRB prefer?" and validates
that the trained module contributes positively at its learned gate value.

In [ ]:
try:
    gate_sweep_data = []
    gsm_mini = gsm8k["test"].select(range(20))

    for g_val in [0.00, 0.01, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]:
        nhrt_model.model.layers[TARGET_LAYER].reason_gate.data.fill_(g_val)
        acc = run_eval(nhrt_model, gsm_mini, desc=f"gate={g_val}")
        gate_sweep_data.append((g_val, acc))
        print(f"Gate={g_val:.2f} | Accuracy={acc:.2%}")

    # Restore trained gate
    nhrt_model.model.layers[TARGET_LAYER].reason_gate.data.fill_(trained_gate)
    print(f"\nTrained gate restored: {trained_gate:.4f}")
    print(f"Peak accuracy at gate: {max(gate_sweep_data, key=lambda x: x[1])}")
except Exception as e:
    print(f"Gate sweep skipped: {e}")

## 20. Interpretation Guide & Next Steps

### Reading the Results

| Outcome | What it means | What to do next |
|---|---|---|
| NHRT > LoRA (GSM8K) | Architecture matters, not just parameters | Scale N_TRAIN → 500, increase EPOCHS → 5, submit workshop paper |
| NHRT ≤ LoRA | HRB not adding value beyond capacity | Try stronger DeepReasoner, or adaptive step count |
| Multi > Single | Reasoning modules compound across depth | Scale to all 24 layers with gradient checkpointing on A100 |
| Multi ≤ Single | Layer interference | Investigate: do gates at 6 and 18 open? If not, those layers are wasted |
| DeepReasoner > HRB | Iterative loop is the right idea | Make step count adaptive (Graves 2016 ACT mechanism) |
| SVAMP gap > 0 | Reasoning improvement generalises | Strong publication signal — add ARC-Challenge next |

### Recommended Immediate Actions (in order)
1. **Record which result row wins** — the gate values and McNemar p-values are the key publication figures.
2. **If NHRT > LoRA with p<0.05**: write the results section of your paper now. This is publishable.
3. **Scale training**: change `N_TRAIN=500`, `EPOCHS=5` and re-run Section 11 only.
4. **Add ARC-Challenge**: `load_dataset("allenai/ai2_arc", "ARC-Challenge")` — answer extraction is multiple-choice (A/B/C/D), requires a small adaptation to `extract_gsm8k_answer`.
5. **Adaptive gate count**: replace `steps=3` in DeepReasoner with a learned scalar — the model learns how many reasoning iterations each token needs.

### TorchTune Formal Recipe (for scaling beyond Colab)
```bash
tune run full_finetune_single_device   --config nhrt_config.yaml   model=qwen2_0_5b   tokenizer=qwen2_tokenizer   dataset=gsm8k   epochs=5   batch_size=4
```
The `NestedDecoderLayer` class can be registered as a custom model component in TorchTune's model catalogue for CLI-driven training at full scale.

---

# PHASE 2 — Fixed & Extended

Sections 1–20 above are the validated Phase 1 pipeline (unchanged). Everything below fixes the
bugs found in the previous run and implements the next-step improvements, with nothing extra:

**Bugs fixed:**
- All new training below uses `torch_dtype="auto"` (bfloat16) end-to-end. The float16 switch
  in the earlier attempt (with no `GradScaler`) is what produced the `avg_loss=nan` runs —
  every reasoning module is now explicitly cast to `model.dtype` right after construction.
- Layer/depth sweep results are persisted into a list of dicts (not overwritten via a reused
  `model` variable), and all accuracy printouts use `.2%` (the earlier run under-reported by 100×).
- ARC-Challenge evaluation now generates enough tokens and uses an anchored regex instead of a
  substring check on the last 10 characters — the previous check could match a stray letter.

**Next-step improvements implemented (from Section 20's own roadmap):**
1. Multi-layer gate investigation (why did gates never move?)
2. Adaptive reasoning depth — ACT-style learned halting (replaces the broken manual depth sweep)
3. DeepReasoner + LoRA joint training (Priority 1)
4. Scaled training data/epochs (Priority 2/3), applied to the winning config
5. Fixed layer sweep
6. ARC-Challenge generalisation, done correctly (Priority 4)
7. Larger evaluation set to address the statistical-power problem from Section 9
8. Consolidated results + efficiency table + interpretation guide

## 21. Multi-Layer Gate Investigation

Section 15's finding — gates at layers 6, 12, and 18 never moved from `init_gate=0.05` — repeated identically in the re-run. Before concluding those layers carry no reasoning signal, rule out the simpler explanation: the gate's effective learning rate was too low for it to move within 3 epochs on 200 samples. This re-runs the same multi-layer config with a higher `init_gate` (0.15) as a diagnostic — if gates still don't move, that's now a genuine, twice-replicated negative result worth reporting as-is.

In [ ]:
MULTI_LAYERS_V2 = [6, 12, 18]
INIT_GATE_HIGH = 0.15

multi_v2_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)

for l_idx in MULTI_LAYERS_V2:
    hrb_l  = HierarchicalReasoningBlock(HIDDEN_SIZE).to(device).to(multi_v2_model.dtype)
    nested = NestedDecoderLayer(multi_v2_model.model.layers[l_idx], hrb_l,
                                 init_gate=INIT_GATE_HIGH).to(device).to(multi_v2_model.dtype)
    multi_v2_model.model.layers[l_idx] = nested

for p in multi_v2_model.parameters():
    p.requires_grad = False
for l_idx in MULTI_LAYERS_V2:
    for p in multi_v2_model.model.layers[l_idx].reasoning_module.parameters():
        p.requires_grad = True
    multi_v2_model.model.layers[l_idx].reason_gate.requires_grad = True

multi_v2_history = train_nhrt(multi_v2_model, gsm_train, epochs=EPOCHS, lr=LR * 0.5, desc="Multi-Layer (gate=0.15)")

multi_v2_gsm_acc, multi_v2_gsm_preds = run_eval(multi_v2_model, gsm_eval, desc="Multi-v2 GSM8K", return_preds=True)

print()
print(f"Multi-Layer (init_gate={INIT_GATE_HIGH}) GSM8K: {multi_v2_gsm_acc:.2%}  (was {multi_gsm_acc:.2%} at init_gate=0.05)")
for l_idx in MULTI_LAYERS_V2:
    g = multi_v2_model.model.layers[l_idx].reason_gate.item()
    print(f"  Layer {l_idx} gate: {g:.4f}  (delta: {g - INIT_GATE_HIGH:+.4f})")

if all(abs(multi_v2_model.model.layers[l].reason_gate.item() - INIT_GATE_HIGH) < 1e-4 for l in MULTI_LAYERS_V2):
    print("\nGates still did not move even at a higher init value — this is now a twice-replicated")
    print("negative result: distributing HRB across layers 6/12/18 is not being used by the model,")
    print("regardless of starting gate value. Report as a genuine finding, not a training artifact.")
else:
    print("\nAt least one gate moved this time — the earlier null result may have been an LR/init")
    print("sensitivity issue rather than a true architectural finding. Worth a proper LR sweep on the gates.")

del multi_v2_model
gc.collect(); torch.cuda.empty_cache()

## 22. Adaptive Reasoning Depth — ACT-Style Halting

Section 20's own recommendation: replace `DeepReasoner`'s fixed `steps=3` with a learned per-token
halting probability (Graves, 2016 — Adaptive Computation Time). Each token decides how many
reasoning iterations it needs, up to `max_steps`. A ponder cost (expected steps + remainder) is
added to the loss with a small weight, so extra pondering has to earn its keep.

In [ ]:
class AdaptiveDeepReasoner(nn.Module):
    """
    ACT-style adaptive reasoning loop. At each step every still-running token gets a halting
    probability p from a linear+sigmoid head. Once a token's cumulative halting probability
    crosses (1 - epsilon) it stops, contributing a remainder weight so that per-token weights
    across steps sum to exactly 1.

    `self.ponder_cost` is populated on every forward() — read it in the training loop and add
    `ponder_weight * ponder_cost` to the task loss. `self.last_avg_steps` reports how many steps
    were actually used on average, for logging.
    """
    def __init__(self, hidden_size=HIDDEN_SIZE, max_steps=6, epsilon=0.01):
        super().__init__()
        self.max_steps = max_steps
        self.epsilon    = epsilon
        self.reason_block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.GELU(),
            nn.Linear(hidden_size, hidden_size)
        )
        self.halting_unit = nn.Linear(hidden_size, 1)
        self.norm = nn.RMSNorm(hidden_size, eps=1e-6)

        self.ponder_cost    = None
        self.last_avg_steps = None

    def forward(self, x):
        batch, seq, _ = x.shape
        dev = x.device

        state             = x
        halting_prob_cum  = torch.zeros(batch, seq, device=dev)
        n_updates         = torch.zeros(batch, seq, device=dev)
        remainders        = torch.zeros(batch, seq, device=dev)
        accumulated_state = torch.zeros_like(x)
        still_running     = torch.ones(batch, seq, device=dev, dtype=torch.bool)

        for _ in range(self.max_steps):
            p = torch.sigmoid(self.halting_unit(state)).squeeze(-1).float()
            running_f = still_running.float()

            would_halt = (halting_prob_cum + p * running_f) >= (1 - self.epsilon)
            halt_now   = would_halt & still_running

            weight = torch.where(halt_now, 1 - halting_prob_cum, p * running_f)

            state = state + self.norm(self.reason_block(state))
            accumulated_state = accumulated_state + weight.unsqueeze(-1).to(state.dtype) * state

            halting_prob_cum = torch.where(still_running, halting_prob_cum + p * running_f, halting_prob_cum)
            n_updates  = n_updates + running_f
            remainders = torch.where(halt_now, weight, remainders)

            still_running = still_running & (~halt_now)
            if not still_running.any():
                break

        if still_running.any():
            weight = torch.where(still_running, 1 - halting_prob_cum, torch.zeros_like(halting_prob_cum))
            accumulated_state = accumulated_state + weight.unsqueeze(-1).to(state.dtype) * state
            remainders = torch.where(still_running, weight, remainders)
            n_updates  = n_updates + still_running.float()

        self.ponder_cost    = (n_updates + remainders).mean()
        self.last_avg_steps = n_updates.mean().item()
        return accumulated_state


PONDER_WEIGHT = 0.01

def train_nhrt_adaptive(model, reasoning_module, train_dataset, epochs=EPOCHS, lr=LR,
                         grad_clip=1.0, warmup_ratio=0.1, batch_size=1,
                         ponder_weight=PONDER_WEIGHT, desc="Training"):
    """Same as train_nhrt, plus: loss += ponder_weight * reasoning_module.ponder_cost"""
    sft_ds    = NHRTSFTDataset(train_dataset, tokenizer)
    loader    = torch.utils.data.DataLoader(sft_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01)

    total_steps  = epochs * len(loader)
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = (get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
                 if HAS_TORCHTUNE else CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=lr * 0.1))

    model.train()
    history, step_history = [], []

    for epoch in range(epochs):
        epoch_loss, epoch_steps = 0.0, []
        for batch in tqdm(loader, desc=f"{desc} E{epoch+1}/{epochs}", leave=False):
            input_ids = batch["input_ids"].to(device)
            labels    = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, labels=labels)
            loss    = outputs.loss
            if reasoning_module.ponder_cost is not None:
                loss = loss + ponder_weight * reasoning_module.ponder_cost.to(loss.dtype)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            if reasoning_module.last_avg_steps is not None:
                epoch_steps.append(reasoning_module.last_avg_steps)

        avg = epoch_loss / len(loader)
        avg_steps = sum(epoch_steps) / len(epoch_steps) if epoch_steps else float("nan")
        history.append(avg); step_history.append(avg_steps)
        print(f"  [{desc}] Epoch {epoch+1}/{epochs}  avg_loss={avg:.4f}  avg_reasoning_steps={avg_steps:.2f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}")

    model.eval()
    return history, step_history


adaptive_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)

adr = AdaptiveDeepReasoner(HIDDEN_SIZE, max_steps=6).to(device).to(adaptive_model.dtype)
adaptive_layer = NestedDecoderLayer(adaptive_model.model.layers[TARGET_LAYER], adr, init_gate=0.05).to(device).to(adaptive_model.dtype)
adaptive_model.model.layers[TARGET_LAYER] = adaptive_layer

for p in adaptive_model.parameters():
    p.requires_grad = False
for p in adaptive_model.model.layers[TARGET_LAYER].reasoning_module.parameters():
    p.requires_grad = True
adaptive_model.model.layers[TARGET_LAYER].reason_gate.requires_grad = True

adaptive_trainable = sum(p.numel() for p in adaptive_model.parameters() if p.requires_grad)
print(f"AdaptiveDeepReasoner trainable: {adaptive_trainable:,}")

adaptive_history, adaptive_steps_history = train_nhrt_adaptive(
    adaptive_model, adr, gsm_train, epochs=EPOCHS, lr=LR, desc="AdaptiveDeepReasoner"
)

adaptive_gsm_acc, adaptive_gsm_preds = run_eval(adaptive_model, gsm_eval, desc="Adaptive GSM8K", return_preds=True)
adaptive_svamp_acc = run_eval(adaptive_model, svamp_gsm_eval, desc="Adaptive SVAMP")
adaptive_gate = adaptive_model.model.layers[TARGET_LAYER].reason_gate.item()

print(f"\nAdaptiveDeepReasoner GSM8K : {adaptive_gsm_acc:.2%}")
print(f"AdaptiveDeepReasoner SVAMP : {adaptive_svamp_acc:.2%}")
print(f"AdaptiveDeepReasoner gate  : {adaptive_gate:.4f}")
print(f"Final avg reasoning steps  : {adaptive_steps_history[-1]:.2f} / {adr.max_steps} max")
print(f"vs fixed-step DeepReasoner : {'adaptive wins' if adaptive_gsm_acc > deep_gsm_acc else 'fixed 3 steps still wins'} "
      f"({adaptive_gsm_acc:.2%} vs {deep_gsm_acc:.2%})")

## 23. DeepReasoner + LoRA Joint Training

Section 17's finding was `NHRT ≈ LoRA` (p=1.0000, not significant) — a tie, not a loss, but the
open question is whether the two mechanisms are substitutes or complements. This applies LoRA to
layer 12's projections first, then splices `DeepReasoner` into the resulting PEFT-wrapped layer,
training both jointly at `r=16` (half of the standalone LoRA control's `r=32`) so the combined
budget stays comparable rather than just adding more parameters.

In [ ]:
joint_lora_targets = [
    f"model.layers.{TARGET_LAYER}.self_attn.q_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.k_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.v_proj",
    f"model.layers.{TARGET_LAYER}.self_attn.o_proj",
    f"model.layers.{TARGET_LAYER}.mlp.gate_proj",
    f"model.layers.{TARGET_LAYER}.mlp.up_proj",
    f"model.layers.{TARGET_LAYER}.mlp.down_proj",
]

joint_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)
joint_lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=joint_lora_targets,
                                lora_dropout=0.0, bias="none", task_type="CAUSAL_LM")
joint_model = get_peft_model(joint_base, joint_lora_config)

# PEFT wraps as PeftModel -> base_model (LoraModel) -> model -> layers — one level deeper
# than the raw HF model, hence .base_model.model.model.layers here.
inner_layers = joint_model.base_model.model.model.layers

dr_joint = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(joint_model.dtype)
inner_layers[TARGET_LAYER] = NestedDecoderLayer(
    inner_layers[TARGET_LAYER], dr_joint, init_gate=0.05
).to(device).to(joint_model.dtype)

for p in inner_layers[TARGET_LAYER].reasoning_module.parameters():
    p.requires_grad = True
inner_layers[TARGET_LAYER].reason_gate.requires_grad = True

joint_trainable = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
joint_total     = sum(p.numel() for p in joint_model.parameters())
print(f"Joint DeepReasoner+LoRA trainable: {joint_trainable:,} / {joint_total:,} ({100*joint_trainable/joint_total:.3f}%)")

with torch.no_grad():
    _inp = tokenizer("What is 3 + 4?", return_tensors="pt").to(device)
    _out = joint_model.generate(**_inp, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    print("Functional check:", tokenizer.decode(_out[0], skip_special_tokens=True))

joint_history = train_nhrt(joint_model, gsm_train, epochs=EPOCHS, lr=LR, desc="DeepReasoner+LoRA")

joint_gsm_acc, joint_gsm_preds = run_eval(joint_model, gsm_eval, desc="Joint GSM8K", return_preds=True)
joint_svamp_acc = run_eval(joint_model, svamp_gsm_eval, desc="Joint SVAMP")
joint_gate = inner_layers[TARGET_LAYER].reason_gate.item()

print(f"\nDeepReasoner+LoRA GSM8K : {joint_gsm_acc:.2%}")
print(f"DeepReasoner+LoRA SVAMP : {joint_svamp_acc:.2%}")
print(f"DeepReasoner+LoRA gate  : {joint_gate:.4f}  (delta: {joint_gate-0.05:+.4f})")
print(f"vs standalone LoRA        : {'joint wins' if joint_gsm_acc > lora_gsm_acc else 'no improvement over LoRA alone'} "
      f"({joint_gsm_acc:.2%} vs {lora_gsm_acc:.2%})")
print(f"vs standalone DeepReasoner: {'joint wins' if joint_gsm_acc > deep_gsm_acc else 'no improvement over DeepReasoner alone'} "
      f"({joint_gsm_acc:.2%} vs {deep_gsm_acc:.2%})")

mcnemar_test(joint_gsm_preds, lora_gsm_preds, "Joint", "LoRA")
mcnemar_test(joint_gsm_preds, deep_gsm_preds, "Joint", "DeepReasoner")

## 24. Scaled Training — N_TRAIN 200→1000, EPOCHS 3→5

Re-runs the strongest architecture from Sections 21–23 at the scale Section 20 recommended.
`BEST_MODULE` is a one-line switch — set it based on which of hrb / deep / adaptive / joint
actually won above, rather than trusting a default guess.

In [ ]:
N_TRAIN_SCALED = 1000
EPOCHS_SCALED  = 5

gsm_train_scaled = gsm8k["train"].select(range(N_TRAIN_SCALED))
print(f"Scaled training set: {len(gsm_train_scaled)} samples (was {N_TRAIN})")

BEST_MODULE = "joint"   # change to "hrb", "deep", or "adaptive" based on Sections 21-23's actual winner

def build_candidate(kind):
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)
    if kind == "hrb":
        mod = HierarchicalReasoningBlock(HIDDEN_SIZE).to(device).to(m.dtype)
        m.model.layers[TARGET_LAYER] = NestedDecoderLayer(m.model.layers[TARGET_LAYER], mod, init_gate=0.05).to(device).to(m.dtype)
        for p in m.parameters(): p.requires_grad = False
        for p in m.model.layers[TARGET_LAYER].reasoning_module.parameters(): p.requires_grad = True
        m.model.layers[TARGET_LAYER].reason_gate.requires_grad = True
        return m, mod, None
    if kind == "deep":
        mod = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(m.dtype)
        m.model.layers[TARGET_LAYER] = NestedDecoderLayer(m.model.layers[TARGET_LAYER], mod, init_gate=0.05).to(device).to(m.dtype)
        for p in m.parameters(): p.requires_grad = False
        for p in m.model.layers[TARGET_LAYER].reasoning_module.parameters(): p.requires_grad = True
        m.model.layers[TARGET_LAYER].reason_gate.requires_grad = True
        return m, mod, None
    if kind == "adaptive":
        mod = AdaptiveDeepReasoner(HIDDEN_SIZE, max_steps=6).to(device).to(m.dtype)
        m.model.layers[TARGET_LAYER] = NestedDecoderLayer(m.model.layers[TARGET_LAYER], mod, init_gate=0.05).to(device).to(m.dtype)
        for p in m.parameters(): p.requires_grad = False
        for p in m.model.layers[TARGET_LAYER].reasoning_module.parameters(): p.requires_grad = True
        m.model.layers[TARGET_LAYER].reason_gate.requires_grad = True
        return m, mod, "adaptive"
    if kind == "joint":
        pm = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, target_modules=joint_lora_targets,
                                           lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"))
        layers = pm.base_model.model.model.layers
        mod = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(pm.dtype)
        layers[TARGET_LAYER] = NestedDecoderLayer(layers[TARGET_LAYER], mod, init_gate=0.05).to(device).to(pm.dtype)
        for p in layers[TARGET_LAYER].reasoning_module.parameters(): p.requires_grad = True
        layers[TARGET_LAYER].reason_gate.requires_grad = True
        return pm, mod, None
    raise ValueError(kind)

scaled_model, scaled_module, scaled_kind = build_candidate(BEST_MODULE)
scaled_trainable = sum(p.numel() for p in scaled_model.parameters() if p.requires_grad)
print(f"Scaled run — architecture: {BEST_MODULE}  |  trainable params: {scaled_trainable:,}")

if scaled_kind == "adaptive":
    scaled_history, scaled_steps_history = train_nhrt_adaptive(
        scaled_model, scaled_module, gsm_train_scaled, epochs=EPOCHS_SCALED, lr=LR, desc=f"Scaled-{BEST_MODULE}"
    )
else:
    scaled_history = train_nhrt(scaled_model, gsm_train_scaled, epochs=EPOCHS_SCALED, lr=LR, desc=f"Scaled-{BEST_MODULE}")

scaled_gsm_acc, scaled_gsm_preds = run_eval(scaled_model, gsm_eval, desc="Scaled GSM8K", return_preds=True)
scaled_svamp_acc = run_eval(scaled_model, svamp_gsm_eval, desc="Scaled SVAMP")

print(f"\n[{BEST_MODULE}] Scaled ({N_TRAIN_SCALED} samples, {EPOCHS_SCALED} epochs) GSM8K : {scaled_gsm_acc:.2%}")
print(f"[{BEST_MODULE}] Scaled SVAMP : {scaled_svamp_acc:.2%}")
print(f"[{BEST_MODULE}] vs the {N_TRAIN}-sample/{EPOCHS}-epoch NHRT reference: "
      f"{(scaled_gsm_acc - nhrt_gsm_acc)*100:+.2f}pp GSM8K")

## 25. Layer Sweep (Fixed)

Same idea as the earlier attempt, fixed: consistent `bfloat16` throughout (no manual `float16`),
explicit `.to(model.dtype)` casts, results persisted into a list of dicts rather than a
loop-overwritten `model` variable, and `.2%` formatting throughout.

In [ ]:
import pandas as pd

LAYER_SWEEP = [4, 8, 12, 16, 20, 23]
layer_sweep_results = []

for layer_idx in LAYER_SWEEP:
    torch.cuda.empty_cache(); gc.collect()

    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)

    dr_l = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(m.dtype)
    wrapped = NestedDecoderLayer(m.model.layers[layer_idx], dr_l, init_gate=0.05).to(device).to(m.dtype)
    m.model.layers[layer_idx] = wrapped

    for p in m.parameters(): p.requires_grad = False
    for p in m.model.layers[layer_idx].reasoning_module.parameters(): p.requires_grad = True
    m.model.layers[layer_idx].reason_gate.requires_grad = True

    train_nhrt(m, gsm_train, epochs=2, lr=LR, desc=f"Layer {layer_idx}")
    acc = run_eval(m, gsm_eval, desc=f"Layer {layer_idx} eval")
    gate = m.model.layers[layer_idx].reason_gate.item()

    layer_sweep_results.append({"layer": layer_idx, "accuracy": acc, "gate": gate, "gate_delta": gate - 0.05})
    print(f"Layer {layer_idx:>2} | Accuracy={acc:.2%} | Gate={gate:.4f} (delta {gate-0.05:+.4f})")

    del m, dr_l, wrapped
    gc.collect(); torch.cuda.empty_cache()

layer_sweep_df = pd.DataFrame(layer_sweep_results)
print("\nLAYER SWEEP RESULTS")
display(layer_sweep_df)

best_layer_row = layer_sweep_df.loc[layer_sweep_df["accuracy"].idxmax()]
print(f"\nBest layer: {int(best_layer_row['layer'])}  (accuracy={best_layer_row['accuracy']:.2%}, "
      f"gate_delta={best_layer_row['gate_delta']:+.4f})")
print(f"Reference — TARGET_LAYER={TARGET_LAYER} (Phase 1 winner) accuracy: {nhrt_gsm_acc:.2%} for comparison")

## 26. ARC-Challenge Generalisation (Fixed)

GSM8K/SVAMP are both arithmetic word problems; ARC-Challenge is multiple-choice science reasoning
— a genuinely different task family. The earlier attempt's evaluator generated only 3 tokens and
matched with a raw substring check on the last 10 characters, which can match a stray letter by
chance. This generates enough tokens for an explicit "Answer: X" pattern and extracts with an
anchored regex, falling back to a tighter tail-window check only if that fails.

In [ ]:
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")
N_ARC_EVAL = min(300, len(arc["test"]))
arc_eval = arc["test"].select(range(N_ARC_EVAL))
print(f"ARC-Challenge test samples: {N_ARC_EVAL}")

def format_arc_prompt(sample):
    q = sample["question"]
    choices, labels = sample["choices"]["text"], sample["choices"]["label"]
    opts = "\n".join(f"{l}. {t}" for l, t in zip(labels, choices))
    return f"{q}\n{opts}\nAnswer:"

def extract_arc_answer(text):
    m = re.search(r"Answer:\s*\(?([A-E])\)?", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m2 = re.search(r"\b([A-E])\b", text.strip()[-15:])
    return m2.group(1).upper() if m2 else None

def run_eval_arc(model, dataset, max_new_tokens=16, desc="ARC Eval", return_preds=False):
    model.eval()
    correct, preds = 0, []
    for sample in tqdm(dataset, desc=desc, leave=False):
        prompt = format_arc_prompt(sample)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        gen  = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extract_arc_answer(gen)
        true = sample["answerKey"].upper()
        ok   = (pred == true)
        correct += int(ok)
        preds.append(ok)
    acc = correct / len(dataset)
    return (acc, preds) if return_preds else acc


baseline_arc_acc, baseline_arc_preds = run_eval_arc(baseline_model, arc_eval, desc="Baseline ARC", return_preds=True)
scaled_arc_acc, scaled_arc_preds     = run_eval_arc(scaled_model,   arc_eval, desc=f"{BEST_MODULE} ARC", return_preds=True)

print()
print("=" * 55)
print("ARC-CHALLENGE GENERALISATION (out-of-domain, multiple-choice)")
print("=" * 55)
print(f"Baseline           : {baseline_arc_acc:.2%}")
print(f"{BEST_MODULE:<20}: {scaled_arc_acc:.2%}   (delta: {(scaled_arc_acc-baseline_arc_acc)*100:+.2f}pp)")
mcnemar_test(scaled_arc_preds, baseline_arc_preds, BEST_MODULE, "Baseline")

## 27. Larger Evaluation Set — Statistical Power

Section 9's p-values (0.18–0.75 in Phase 1; the re-run's NHRT-vs-Baseline did clear p<0.05, but
NHRT-vs-LoRA, Multi-vs-Single, and DeepReasoner-vs-HRB all stayed at p=1.0 exactly) are very
plausibly a sample-size problem — n=100 with 5–8pp effects and few disagreement pairs can't
reliably separate real ties from real effects. This re-evaluates at n=300.

In [ ]:
N_EVAL_LARGE = 300
gsm_eval_large = gsm8k["test"].select(range(N_EVAL_LARGE))
print(f"Large-scale eval set: {N_EVAL_LARGE} samples (was {N_EVAL})")

baseline_large_acc, baseline_large_preds = run_eval(baseline_model, gsm_eval_large, desc="Baseline (n=300)", return_preds=True)
scaled_large_acc, scaled_large_preds     = run_eval(scaled_model,   gsm_eval_large, desc=f"{BEST_MODULE} (n=300)", return_preds=True)

print()
print("=" * 60)
print(f"LARGE-SCALE EVAL (n={N_EVAL_LARGE})")
print("=" * 60)
print(f"Baseline        : {baseline_large_acc:.2%}")
print(f"{BEST_MODULE:<16}: {scaled_large_acc:.2%}   (delta: {(scaled_large_acc-baseline_large_acc)*100:+.2f}pp)")
mcnemar_test(scaled_large_preds, baseline_large_preds, BEST_MODULE, "Baseline")

## 28. Consolidated Results, Efficiency & Interpretation

In [ ]:
print("=" * 85)
print("PHASE 2 CONSOLIDATED RESULTS")
print("=" * 85)
print(f"{'Configuration':<34}{'GSM8K':>10}{'SVAMP':>10}{'ARC':>10}{'Δ GSM8K':>12}")
print("-" * 85)

phase2_rows = [
    ("Baseline (frozen)",                 baseline_gsm_acc, baseline_svamp_acc, baseline_arc_acc),
    (f"LoRA L{TARGET_LAYER} (Phase 1)",   lora_gsm_acc,     lora_svamp_acc,     None),
    (f"HRB L{TARGET_LAYER} (Phase 1)",    nhrt_gsm_acc,     nhrt_svamp_acc,     None),
    (f"DeepReasoner L{TARGET_LAYER} (P1)",deep_gsm_acc,     deep_svamp_acc,     None),
    ("Multi-Layer (gate=0.15)",           multi_v2_gsm_acc, None,               None),
    ("AdaptiveDeepReasoner (ACT)",        adaptive_gsm_acc, adaptive_svamp_acc, None),
    ("DeepReasoner + LoRA (joint)",       joint_gsm_acc,    joint_svamp_acc,    None),
    (f"Scaled best [{BEST_MODULE}]",      scaled_gsm_acc,   scaled_svamp_acc,   scaled_arc_acc),
]

for name, gsm, sv, arc_acc in phase2_rows:
    dg = (gsm - baseline_gsm_acc) * 100
    sv_str  = f"{sv:.2%}" if sv is not None else "  n/a"
    arc_str = f"{arc_acc:.2%}" if arc_acc is not None else "  n/a"
    print(f"{name:<34}{gsm:>10.2%}{sv_str:>10}{arc_str:>10}{dg:>+11.2f}pp")

print()
print("Efficiency (trainable params, on the winning scaled config):")
scaled_trainable_final = sum(p.numel() for p in scaled_model.parameters() if p.requires_grad)
scaled_total_final     = sum(p.numel() for p in scaled_model.parameters())
print(f"  {BEST_MODULE}: {scaled_trainable_final:,} / {scaled_total_final:,} "
      f"({100*scaled_trainable_final/scaled_total_final:.3f}% trainable)")

print()
print("Key comparisons:")
print(f"  Joint vs standalone LoRA        : {'joint wins' if joint_gsm_acc > lora_gsm_acc else 'LoRA still wins/ties'}")
print(f"  Joint vs standalone DeepReasoner : {'joint wins' if joint_gsm_acc > deep_gsm_acc else 'DeepReasoner alone still wins/ties'}")
print(f"  Adaptive vs fixed-step DeepReasoner: {'adaptive wins' if adaptive_gsm_acc > deep_gsm_acc else 'fixed 3 steps still wins/ties'} "
      f"(avg steps used: {adaptive_steps_history[-1]:.2f}/{adr.max_steps})")
print(f"  n=300 significance              : see Section 27 p-value above")
print(f"  ARC-Challenge (out-of-domain)   : {'generalises beyond arithmetic' if scaled_arc_acc > baseline_arc_acc else 'no transfer to non-arithmetic reasoning'}")

results_final_rows = [
    {"config": name, "gsm8k": gsm, "svamp": sv, "arc": arc_acc}
    for name, gsm, sv, arc_acc in phase2_rows
]
with open("nhrt_phase2_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["config", "gsm8k", "svamp", "arc"])
    writer.writeheader()
    writer.writerows(results_final_rows)
print("\nSaved: nhrt_phase2_results.csv")

### Reading the Phase 2 Results

| Outcome | What it means | What to do next |
|---|---|---|
| Multi-layer gates still frozen at 0.15 init | Twice-replicated null — those layers don't use HRB | Report as a genuine finding; investigate per-gate LR separately if pursuing further |
| Joint > standalone LoRA and DeepReasoner | Mechanisms are complementary | Strongest publication result — lead with this |
| Joint ≈ LoRA alone | LoRA absorbs the reasoning benefit | Valid negative result — still worth reporting |
| Adaptive > fixed-step DeepReasoner | Learned depth beats a hand-picked constant | Report avg-steps-used as a figure |
| p < 0.05 at n=300 | Effect is real, not sampling noise | Resolves the Phase 1 "not significant" caveat |
| ARC-Challenge flat/negative | Gains are arithmetic-specific | Scope paper claims to math word problems, not "general reasoning" |
| ARC-Challenge improves | Reasoning gain generalises past arithmetic | Consider adding MATH or CommonsenseQA next |

### Immediate next actions
1. Run top-to-bottom in Colab (GPU required, same T4/A100 setup).
2. Set `BEST_MODULE` in Section 24 based on what Sections 21–23 actually show — don't trust the default.
3. If joint training is unstable, lower `joint_lora_config`'s `r` to 8 or drop `LR` to `1e-4` for that run only.
4. For the next scale-up beyond Colab: register `NestedDecoderLayer` as a custom TorchTune model component and use the formal `tune run full_finetune_single_device` CLI recipe for a full 24-layer, gradient-checkpointed run on an A100.

---
# Phase 3 — Closing the Gaps Before Publication

Addresses, in priority order: (1) the unverified Adaptive-vs-DeepReasoner comparison,
(2) run-to-run variance, (3) the ARC format-floor problem, (4) the Layer 8 lead,
and (5) DPO as a follow-up refinement on top of the AdaptiveDeepReasoner SFT checkpoint.

QLoRA is intentionally **not** used here — see the note in Section 34 for why.

## 29. Significance Test: Adaptive vs Fixed-Step DeepReasoner
Your best result (12% vs 11%) has never been tested for significance. Fixing that first,
since everything downstream assumes this gap is real.

In [ ]:
print("McNemar's test: AdaptiveDeepReasoner vs fixed-step DeepReasoner (GSM8K)")
mcnemar_test(adaptive_gsm_preds, deep_gsm_preds, "Adaptive", "DeepReasoner")

print()
print("McNemar's test: AdaptiveDeepReasoner vs Baseline (GSM8K)")
mcnemar_test(adaptive_gsm_preds, baseline_gsm_preds, "Adaptive", "Baseline")


## 30. Seed Variance Harness (fixes Drawback #1)
HRB swung from 11% to 8% across identical-hyperparameter runs, flipping a significant
result to a non-significant one. Until we know each architecture's mean ± std across seeds,
no single run's number can be trusted. This runs baseline, HRB, DeepReasoner, and Adaptive
across multiple seeds at n=300 (per the Section 27 lesson that n=100 isn't stable).

**Note:** this is compute-heavy (N_SEEDS × 3 architectures × full training). Reduce
`N_SEEDS` or `EPOCHS` if you need a faster pass first.

In [ ]:
import numpy as np

N_SEEDS = 3
SEEDS = [42, 123, 2024][:N_SEEDS]
VARIANCE_CANDIDATES = ["hrb", "deep", "adaptive"]   # baseline is deterministic (no training), evaluated once
N_EVAL_VAR = N_EVAL_LARGE   # 300 — do not go back to n=100 for this

def run_seeded_candidate(kind, seed):
    torch.manual_seed(seed)
    m, mod, special = build_candidate(kind)
    if special == "adaptive":
        train_nhrt_adaptive(m, mod, gsm_train, epochs=EPOCHS, lr=LR, desc=f"{kind}-seed{seed}")
    else:
        train_nhrt(m, gsm_train, epochs=EPOCHS, lr=LR, desc=f"{kind}-seed{seed}")
    acc, preds = run_eval(m, gsm_eval_large, desc=f"{kind}-seed{seed} eval(n={N_EVAL_VAR})", return_preds=True)
    del m, mod
    gc.collect(); torch.cuda.empty_cache()
    return acc, preds

variance_results = {}
variance_preds   = {}
for kind in VARIANCE_CANDIDATES:
    accs, preds_by_seed = [], []
    for seed in SEEDS:
        acc, preds = run_seeded_candidate(kind, seed)
        accs.append(acc)
        preds_by_seed.append(preds)
        print(f"[{kind}] seed={seed} -> GSM8K acc={acc:.2%}")
    variance_results[kind] = accs
    variance_preds[kind]   = preds_by_seed

print()
print("=" * 65)
print(f"VARIANCE SUMMARY (n_eval={N_EVAL_VAR}, seeds={SEEDS})")
print("=" * 65)
print(f"Baseline (no training, deterministic): {baseline_large_acc:.2%}")
for kind, accs in variance_results.items():
    arr = np.array(accs)
    print(f"{kind:<10}: mean={arr.mean():.2%}  std={arr.std():.2%}  "
          f"range=[{arr.min():.2%}, {arr.max():.2%}]  n_seeds={len(accs)}")

print()
print("Interpretation: if a candidate's std is comparable to its mean delta over baseline,")
print("that candidate's apparent advantage is not yet distinguishable from seed noise.")


## 31. Fixed ARC-Challenge Evaluation: Log-Likelihood Choice Scoring (fixes Drawback #5)
Free-form generation + regex extraction fails near-universally for a non-instruction-tuned
0.5B model, hence the 2% floor for both baseline and treatment. Scoring each answer choice by
the model's log-likelihood of that continuation sidesteps the format-following requirement
entirely and is the standard approach for small base models on multiple-choice tasks.

In [ ]:
def score_choice_loglik(model, question_prompt, choice_text):
    """Total log-likelihood of choice_text's tokens, continuing question_prompt."""
    full = question_prompt + " " + choice_text
    prompt_ids = tokenizer(question_prompt, return_tensors="pt").input_ids.to(device)
    full_ids   = tokenizer(full, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = model(full_ids)
    logits  = out.logits[:, :-1, :]
    targets = full_ids[:, 1:]
    logprobs = F.log_softmax(logits.float(), dim=-1)
    token_logprobs = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    choice_len = full_ids.shape[1] - prompt_ids.shape[1]
    if choice_len <= 0:
        return float("-inf")
    return token_logprobs[0, -choice_len:].sum().item()

def format_arc_question_only(sample):
    return sample["question"] + "\nAnswer:"

def run_eval_arc_loglik(model, dataset, desc="ARC LogLik Eval", return_preds=False):
    model.eval()
    correct, preds = 0, []
    for sample in tqdm(dataset, desc=desc, leave=False):
        prompt = format_arc_question_only(sample)
        choices, labels = sample["choices"]["text"], sample["choices"]["label"]
        scores = [score_choice_loglik(model, prompt, c) for c in choices]
        pred_label = labels[int(np.argmax(scores))]
        true = sample["answerKey"].upper()
        ok = (pred_label.upper() == true)
        correct += int(ok)
        preds.append(ok)
    acc = correct / len(dataset)
    return (acc, preds) if return_preds else acc


baseline_arc_ll_acc, baseline_arc_ll_preds = run_eval_arc_loglik(
    baseline_model, arc_eval, desc="Baseline ARC (log-lik)", return_preds=True)
scaled_arc_ll_acc, scaled_arc_ll_preds = run_eval_arc_loglik(
    scaled_model, arc_eval, desc=f"{BEST_MODULE} ARC (log-lik)", return_preds=True)

print()
print("=" * 65)
print("ARC-CHALLENGE — LOG-LIKELIHOOD CHOICE SCORING (fixes format-floor issue)")
print("=" * 65)
print(f"Baseline           : {baseline_arc_ll_acc:.2%}  (was {baseline_arc_acc:.2%} under free-gen+regex)")
print(f"{BEST_MODULE:<20}: {scaled_arc_ll_acc:.2%}  (was {scaled_arc_acc:.2%} under free-gen+regex)")
mcnemar_test(scaled_arc_ll_preds, baseline_arc_ll_preds, BEST_MODULE, "Baseline")


## 32. Layer 8 Follow-Up: Full Run (fixes Drawback / Achievement #4)
The lighter 2-epoch sweep suggested Layer 8 (12%) may beat Layer 12 (11%). Confirming with a
full-length run before considering changing `TARGET_LAYER`.

In [ ]:
LAYER8 = 8
m8  = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(device)
dr8 = DeepReasoner(HIDDEN_SIZE, steps=3).to(device).to(m8.dtype)
m8.model.layers[LAYER8] = NestedDecoderLayer(m8.model.layers[LAYER8], dr8, init_gate=0.05).to(device).to(m8.dtype)

for p in m8.parameters():
    p.requires_grad = False
for p in m8.model.layers[LAYER8].reasoning_module.parameters():
    p.requires_grad = True
m8.model.layers[LAYER8].reason_gate.requires_grad = True

train_nhrt(m8, gsm_train, epochs=EPOCHS, lr=LR, desc=f"DeepReasoner Layer{LAYER8} (full run)")

layer8_gsm_acc, layer8_gsm_preds = run_eval(m8, gsm_eval, desc=f"Layer{LAYER8} GSM8K", return_preds=True)
layer8_svamp_acc = run_eval(m8, svamp_gsm_eval, desc=f"Layer{LAYER8} SVAMP")
layer8_gate = m8.model.layers[LAYER8].reason_gate.item()

print(f"\nLayer {LAYER8} (full run) GSM8K : {layer8_gsm_acc:.2%}  (2-epoch sweep showed 12%)")
print(f"Layer {LAYER8} (full run) SVAMP : {layer8_svamp_acc:.2%}")
print(f"Layer {LAYER8} gate            : {layer8_gate:.4f}  (delta {layer8_gate-0.05:+.4f})")
print(f"vs Layer {TARGET_LAYER} DeepReasoner (full run): {deep_gsm_acc:.2%}")

mcnemar_test(layer8_gsm_preds, deep_gsm_preds, f"Layer{LAYER8}", f"Layer{TARGET_LAYER}")

del m8, dr8
gc.collect(); torch.cuda.empty_cache()


## 33. DPO on Self-Generated Preference Pairs (AdaptiveDeepReasoner)
Applied only after Sections 29–32, since layering a second training stage onto an
unverified config just compounds noise.

**Why DPO fits here specifically:** we don't have human preference labels, so we generate
them ourselves — sample several completions per training question from the already-SFT'd
`adaptive_model`, keep ones that reach the correct final answer as *chosen* and incorrect
ones as *rejected* (rejection-sampling / STaR-style). Among correct completions we prefer the
**shortest**, which directly reinforces the efficiency story ACT is already telling (2.05 avg
steps vs fixed-3). Only the reasoning module + gate stay trainable — same trainable-parameter
scope as the SFT stage — with a frozen deep copy of the SFT checkpoint as the DPO reference
policy.

In [ ]:
import copy, random

DPO_N_QUESTIONS   = 150     # candidate questions drawn from the scaled pool
DPO_SAMPLES_PER_Q = 4       # sampled completions per question
DPO_BETA          = 0.1
DPO_LR            = 5e-5
DPO_EPOCHS        = 2

def sample_completion(model, question, max_new_tokens=96, temperature=0.8):
    inputs = tokenizer(question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperature, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def build_preference_pairs(model, pool, n_questions=DPO_N_QUESTIONS, k=DPO_SAMPLES_PER_Q):
    pairs = []
    subset = pool.select(range(min(n_questions, len(pool))))
    for sample in tqdm(subset, desc="Building DPO pairs"):
        q, true_ans = sample["question"], sample["answer"]
        true_num = extract_gsm8k_answer(true_ans)
        completions = [sample_completion(model, q) for _ in range(k)]
        scored  = [(c, extract_gsm8k_answer(c) == true_num) for c in completions]
        correct = [c for c, ok in scored if ok]
        wrong   = [c for c, ok in scored if not ok]
        if correct and wrong:
            chosen   = min(correct, key=len)   # shortest correct trace -> reinforces efficient halting
            rejected = wrong[0]
            pairs.append({"question": q, "chosen": chosen, "rejected": rejected})
    return pairs

dpo_pairs = build_preference_pairs(adaptive_model, gsm_train_scaled)
print(f"Built {len(dpo_pairs)} preference pairs out of {DPO_N_QUESTIONS} candidate questions "
      f"({len(dpo_pairs)/DPO_N_QUESTIONS:.0%} yield — questions with both a correct and incorrect sample).")

# Frozen reference = the SFT-trained adaptive_model, untouched from here on
adaptive_ref = copy.deepcopy(adaptive_model).to(device)
for p in adaptive_ref.parameters():
    p.requires_grad = False
adaptive_ref.eval()

def sequence_logprob(model, question, answer):
    q_ids    = tokenizer(question, return_tensors="pt").input_ids.to(device)
    full_ids = tokenizer(question + "\n" + answer, return_tensors="pt").input_ids.to(device)
    out = model(full_ids)
    logits  = out.logits[:, :-1, :]
    targets = full_ids[:, 1:]
    logprobs = F.log_softmax(logits.float(), dim=-1)
    token_lp = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    ans_len  = full_ids.shape[1] - q_ids.shape[1]
    return token_lp[0, -ans_len:].sum() if ans_len > 0 else torch.tensor(0.0, device=device)

dpo_optimizer = AdamW(filter(lambda p: p.requires_grad, adaptive_model.parameters()),
                       lr=DPO_LR, weight_decay=0.01)

adaptive_model.train()
for epoch in range(DPO_EPOCHS):
    random.shuffle(dpo_pairs)
    epoch_loss, n_correct_pref = 0.0, 0
    for pair in tqdm(dpo_pairs, desc=f"DPO E{epoch+1}/{DPO_EPOCHS}"):
        q, chosen, rejected = pair["question"], pair["chosen"], pair["rejected"]

        pol_chosen   = sequence_logprob(adaptive_model, q, chosen)
        pol_rejected = sequence_logprob(adaptive_model, q, rejected)
        with torch.no_grad():
            ref_chosen   = sequence_logprob(adaptive_ref, q, chosen)
            ref_rejected = sequence_logprob(adaptive_ref, q, rejected)

        pi_logratio  = pol_chosen - pol_rejected
        ref_logratio = ref_chosen - ref_rejected
        dpo_logits   = DPO_BETA * (pi_logratio - ref_logratio)
        loss = -F.logsigmoid(dpo_logits)

        dpo_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in adaptive_model.parameters() if p.requires_grad], 1.0)
        dpo_optimizer.step()

        epoch_loss += loss.item()
        n_correct_pref += int(dpo_logits.item() > 0)

    print(f"  [DPO] Epoch {epoch+1}/{DPO_EPOCHS}  avg_loss={epoch_loss/len(dpo_pairs):.4f}  "
          f"pref_accuracy={n_correct_pref/len(dpo_pairs):.2%}")

adaptive_model.eval()

dpo_gsm_acc, dpo_gsm_preds = run_eval(adaptive_model, gsm_eval, desc="Adaptive+DPO GSM8K", return_preds=True)
dpo_svamp_acc = run_eval(adaptive_model, svamp_gsm_eval, desc="Adaptive+DPO SVAMP")
dpo_gate = adaptive_model.model.layers[TARGET_LAYER].reason_gate.item()

print(f"\nAdaptiveDeepReasoner + DPO GSM8K : {dpo_gsm_acc:.2%}  (pre-DPO SFT-only was {adaptive_gsm_acc:.2%})")
print(f"AdaptiveDeepReasoner + DPO SVAMP : {dpo_svamp_acc:.2%}  (pre-DPO SFT-only was {adaptive_svamp_acc:.2%})")
print(f"Gate after DPO: {dpo_gate:.4f}")

mcnemar_test(dpo_gsm_preds, adaptive_gsm_preds, "Adaptive+DPO", "Adaptive-SFT-only")


## 34. Note on QLoRA — Why It's Skipped by Default
QLoRA's entire value proposition is fitting a **large** model's frozen weights into limited
GPU memory via 4-bit quantization. `Qwen/Qwen2.5-0.5B` already fits trivially in fp16/bf16 on
any modern GPU — memory was never the bottleneck here. Adding 4-bit quantization on top of a
model this small only introduces quantization noise on top of the variance problem Section 30
is already trying to fix, with no compensating benefit.

Standard LoRA (already used in Sections 14 and 23) is the right tool at this scale. The stub
below is here for if/when you move to a meaningfully larger base model.

In [ ]:
# Only relevant if MODEL_NAME changes to something meaningfully bigger (e.g. Qwen2.5-1.5B/3B+).
USE_QLORA = False

if USE_QLORA:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    qlora_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
    qlora_base = prepare_model_for_kbit_training(qlora_base)
    qlora_config = LoraConfig(r=16, lora_alpha=32, target_modules=joint_lora_targets,
                               lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    qlora_model = get_peft_model(qlora_base, qlora_config)
    print("QLoRA model ready — attach a reasoning module the same way as Section 23's joint_model.")
else:
    print("QLoRA skipped — see note above. Standard LoRA (Sections 14/23) is sufficient at 0.5B scale.")
